In [1]:
import json
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch


In [2]:
# ✅ Base chat model
BASE_MODEL = "BioMistral/BioMistral-7B"
DATA_PATH = "fine_tune_data_new.jsonl"

In [3]:
# ---- Load dataset ----
prompts, completions = [], []
with open(DATA_PATH, "r", encoding="utf-8") as f:
    for line in f:
        d = json.loads(line.strip())
        prompts.append(d["prompt"])
        completions.append(d["completion"])

print(f"Loaded {len(prompts)} samples")

Loaded 7475 samples


In [4]:
# ---- Load model ----
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # A10G supports bf16
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

# Leave headroom so auto-sharding never tries CPU/disk
max_mem = {0: "22GiB", "cpu": "0GiB"}  # restrict CPU offload explicitly

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_cfg,
    device_map="auto",        
    max_memory=max_mem,
    low_cpu_mem_usage=True,
)

model.eval()
print("Loaded on:", next(model.parameters()).device)

2025-11-04 00:03:49.413918: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762214629.438103   23589 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762214629.447162   23589 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-11-04 00:03:49.493059: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loaded on: cuda:0


In [6]:
# ---- Chat prompt format for BioMistral ----
def build_prompt(symptoms):
    return f"""
### Instruction:
You are a medical diagnosis assistant. Based on the given patient symptoms, output the most likely disease and the appropriate medical department.

Return answer STRICTLY in this format:

Disease: <disease name>
Department: <department name>

If unsure, output the most reasonable guess based on common medical knowledge.

Symptoms: {symptoms}

### Response:
""".strip()


In [7]:
import re

def format_output(text):
    # normalize lowercase
    text = text.strip()

    disease = "Unknown"
    department = "Unknown"

    # extract disease
    m = re.search(r"Disease\s*:\s*(.+)", text, re.IGNORECASE)
    if m:
        disease = m.group(1).strip()

    # extract department
    m = re.search(r"Department\s*:\s*(.+)", text, re.IGNORECASE)
    if m:
        department = m.group(1).strip()

    return f"Disease: {disease}\nDepartment: {department}"

In [8]:
def generate_baseline(symptoms):
    prompt = build_prompt(symptoms)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=1e-6,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
    
    text = tokenizer.decode(output[0], skip_special_tokens=True)

    if text.startswith(prompt):
        text = text[len(prompt):]

    return format_output(text)

In [9]:
# ---- Run baseline on first N samples ----
N = 50  
baseline_outputs = [generate_baseline(p) for p in prompts[:N]]

# ---- Save result ----
df = pd.DataFrame({
    "symptom_prompt": prompts[:N],
    "baseline_output": baseline_outputs,
    "expected_completion": completions[:N]
})

df.to_csv("baseline_results_biomistral.csv", index=False)
df.head()


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for ope

,symptom_prompt,baseline_output,expected_completion
0,"Given the following symptoms, identify the mos...",Disease: Unknown\nDepartment: Unknown,Disease: allergy\nDepartment: dermatology /
1,"Given the following symptoms, identify the mos...",Disease: Unknown\nDepartment: Unknown,Disease: allergy\nDepartment: dermatology /
2,"Given the following symptoms, identify the mos...",Disease: Allergic reaction Department: Dermato...,Disease: allergy\nDepartment: dermatology /
3,"Given the following symptoms, identify the mos...",Disease: Unknown\nDepartment: Unknown,Disease: allergy\nDepartment: dermatology /
4,"Given the following symptoms, identify the mos...",Disease: Angioedema Department: Emergency Depa...,Disease: allergy\nDepartment: dermatology /


In [11]:
import pandas as pd
import nltk
from nltk.translate.bleu_score import sentence_bleu, corpus_bleu, SmoothingFunction


nltk.download('punkt')
nltk.download('punkt_tab')

df = pd.read_csv("baseline_results_biomistral.csv")

smooth = SmoothingFunction().method3
bleu_scores = []

for ref, base in zip(df["expected_completion"], df["baseline_output"]):
    ref_tokens = nltk.word_tokenize(str(ref))
    base_tokens = nltk.word_tokenize(str(base))

    score = sentence_bleu([ref_tokens], base_tokens, smoothing_function=smooth)
    bleu_scores.append(score)

df["BLEU"] = bleu_scores

print(f"Average BLEU: {df['BLEU'].mean():.4f}")


#df.to_csv("tune_results_with_BLEU.csv", index=False)
#print("Saved to tune_results_with_BLEU.csv")

#df.head()

Average BLEU: 0.1895


[nltk_data] Downloading package punkt to /home/sagemaker-
[nltk_data]     user/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/sagemaker-
[nltk_data]     user/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
